# Bull and Bear Spread Strategies in Options Markets
### Financial Modelling — Course Project under Prof. Mithun Radhakrishna (Group 12)
**Authors:** Jash Bharat Pasad, Jatin Agarwal, Kalabandi Pramith Joy, Sawane Prerna Bharat, Shrey Agarwal

---

## 1. Executive Summary & Strategy Overview
This notebook provides a complete theoretical, numerical, and empirical implementation of two essential vertical options spread strategies:
- **Bull Call Spread (Debit Call Vertical)**: Constructed by buying a lower-strike call ($K_1$) and writing a higher-strike call ($K_2 > K_1$).
- **Bear Put Spread (Debit Put Vertical)**: Constructed by buying a higher-strike put ($K_2$) and writing a lower-strike put ($K_1 < K_2$).

Both strategies provide defined risk and defined reward, capping loss strictly to the net debit paid while lowering the cost basis compared to outright naked option purchases.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.black_scholes import bsm_call_price, bsm_put_price, verify_put_call_parity
from src.strategies.bull_call_spread import BullCallSpread
from src.strategies.bear_put_spread import BearPutSpread
from src.simulation.monte_carlo import MonteCarloEngine
from src.engine.verdict import StrategyVerdictEngine
from src.visualization.charts import (
    plot_bull_call_payoff,
    plot_bear_put_payoff,
    plot_combined_payoff_figure,
    plot_volatility_sensitivity,
    plot_maturity_sensitivity,
    plot_monte_carlo_distribution
)

print('Environment and Quantitative Libraries Loaded Successfully.')

## 2. Baseline Model Parameters
We set the baseline parameters exactly as specified in the course project report:
- Spot Stock Price ($S_0$): **$100.00**
- Time to Expiration ($T$): **0.50 years (6 months)**
- Risk-Free Interest Rate ($r$): **5.0%**
- Annualized Volatility ($\sigma$): **20.0%**
- Bull Call Strikes: $K_1 = 100.00, K_2 = 110.00$
- Bear Put Strikes: $K_1 = 90.00, K_2 = 100.00$

In [ ]:
S0 = 100.0
T = 0.5
r = 0.05
sigma = 0.20

bull = BullCallSpread(S0=S0, K1=100.0, K2=110.0, T=T, r=r, sigma=sigma)
bear = BearPutSpread(S0=S0, K1=90.0, K2=100.0, T=T, r=r, sigma=sigma)

# Table 3: BSM Option Prices
t3_df = pd.DataFrame([
    {'Strategy': 'Bull Call Spread', 'Leg': 'Long Call', 'Strike': f'${bull.K1:.0f}', 'Premium ($)': round(bull.long_call_price, 4)},
    {'Strategy': 'Bull Call Spread', 'Leg': 'Short Call', 'Strike': f'${bull.K2:.0f}', 'Premium ($)': round(bull.short_call_price, 4)},
    {'Strategy': 'Bull Call Spread', 'Leg': 'Net Debit', 'Strike': '-', 'Premium ($)': round(bull.net_debit(), 4)},
    {'Strategy': 'Bear Put Spread', 'Leg': 'Long Put', 'Strike': f'${bear.K2:.0f}', 'Premium ($)': round(bear.long_put_price, 4)},
    {'Strategy': 'Bear Put Spread', 'Leg': 'Short Put', 'Strike': f'${bear.K1:.0f}', 'Premium ($)': round(bear.short_put_price, 4)},
    {'Strategy': 'Bear Put Spread', 'Leg': 'Net Debit', 'Strike': '-', 'Premium ($)': round(bear.net_debit(), 4)}
])
t3_df

## 3. Side-by-Side Strategy Comparison (Tables 1, 2, & 4)
Let us observe the key risk metrics computed from the closed-form solutions.

In [ ]:
t4_df = pd.DataFrame([
    {'Feature': 'Market Outlook', 'Bull Call Spread': 'Moderately bullish', 'Bear Put Spread': 'Moderately bearish'},
    {'Feature': 'Net Debit', 'Bull Call Spread': f'${bull.net_debit():.2f}', 'Bear Put Spread': f'${bear.net_debit():.2f}'},
    {'Feature': 'Maximum Profit', 'Bull Call Spread': f'${bull.max_profit():.2f}', 'Bear Put Spread': f'${bear.max_profit():.2f}'},
    {'Feature': 'Maximum Loss', 'Bull Call Spread': f'-${bull.net_debit():.2f}', 'Bear Put Spread': f'-${bear.net_debit():.2f}'},
    {'Feature': 'Breakeven Price', 'Bull Call Spread': f'${bull.breakeven():.2f}', 'Bear Put Spread': f'${bear.breakeven():.2f}'},
    {'Feature': 'Reward-to-Risk Ratio', 'Bull Call Spread': f'{bull.reward_to_risk():.2f}:1', 'Bear Put Spread': f'{bear.reward_to_risk():.2f}:1'},
    {'Feature': 'Profit Zone', 'Bull Call Spread': f'ST > ${bull.breakeven():.2f}', 'Bear Put Spread': f'ST < ${bear.breakeven():.2f}'}
])
t4_df

## 4. Payoff and Profit Diagrams at Expiration
Here we plot the piecewise linear payoffs and net profit curves, reproducing **Figure 1** from the report.

In [ ]:
fig1 = plot_combined_payoff_figure(bull, bear)
plt.show()

## 5. Sensitivity Analysis
Sensitivity of the strategies to market parameters:
- **Volatility Effect on Bull Call**: Higher IV inflates the net debit paid, thereby lowering the maximum profit ceiling.
- **Maturity Effect on Bear Put**: Modest variation across expiration dates.

In [ ]:
fig_vol = plot_volatility_sensitivity(S0=S0, K1=100.0, K2=110.0, T=T, r=r)
plt.show()

fig_mat = plot_maturity_sensitivity(S0=S0, K1=90.0, K2=100.0, sigma=sigma, r=r)
plt.show()

## 6. Monte Carlo Simulation (10,000 Paths) & Probability of Profit
Using 10,000 simulated Geometric Brownian Motion trajectories, we calculate empirical Probability of Profit (POP), Value at Risk (VaR), and expected return.

In [ ]:
mc_bull = MonteCarloEngine(bull, num_paths=10000, seed=42).run()
mc_bear = MonteCarloEngine(bear, num_paths=10000, seed=42).run()

print(f"Bull Call POP: {mc_bull['probability_of_profit_pct']:.2f}% | Expected PnL: ${mc_bull['expected_total_pnl']:.2f}")
print(f"Bear Put POP:  {mc_bear['probability_of_profit_pct']:.2f}% | Expected PnL: ${mc_bear['expected_total_pnl']:.2f}")

fig_mc = plot_monte_carlo_distribution(mc_bull, strategy_name='Bull Call Spread')
plt.show()